### Milestone 2: Enter the Transformers

- Introduction to the Hugging Face transformers and datasets libraries.
- Learn architecture of BERT/RoBERTa and the concept of attention mechanisms.
- Use pre-trained embedding models to generate context-aware embeddings.
- Zero-shot classification concepts with transformers and SLMs

In [1]:
# %pip install scikit-learn datasets transformers sentence_transformers torch

### Introduction to Hugging Face transformers and datasets

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import torch
import numpy as np

/Users/pratham/Desktop/DL_GenAI_Project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset('csv', data_files='../../data/train.csv')['train']

Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text.
What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51?
Note: We follow zero-indexing here. *

In [4]:
def add_combined_text(example):
    example['combined_text'] = example['prompt'] + ' ' + example['A']
    return example

dataset = dataset.map(add_combined_text)
len(dataset[51]['combined_text'])


614

Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?

In [5]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokenizer.vocab_size

30522

Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.

In [6]:
tokenizer.sep_token_id

102

Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [7]:
texts = list(dataset["prompt"])

tokenized = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

tokenized["input_ids"].shape

torch.Size([2000, 128])

### BERT/RoBERTa Architecture & Attention Mechanisms

A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [8]:
head_dim = 768 / 12
head_dim

64.0

Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [9]:
model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

inputs_row0 = tokenizer(dataset[0]['prompt'], return_tensors='pt')
with torch.no_grad():
    outputs_row0 = model(**inputs_row0)

tuple(outputs_row0.last_hidden_state.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8230.29it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(1, 31, 768)

Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places)

In [10]:
cls_vec = outputs_row0.last_hidden_state[0, 0, :]
cls_vec[:5].sum().item()

-1.2000948190689087

Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).

In [11]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
inputs_t8 = tokenizer(text, return_tensors='pt')
tokens_t8 = tokenizer.convert_ids_to_tokens(inputs_t8['input_ids'][0].tolist())

fusion_idx = tokens_t8.index('fusion')

with torch.no_grad():
    out_t8 = model_attn(**inputs_t8)

attn_weight = out_t8.attentions[-1][0, 0, 0, fusion_idx].item()
attn_weight

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11276.69it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.1024729534983635

### Context-Aware Embeddings 

Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places?
Note: We follow zero-indexing here.

In [12]:
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

emb_prompt = st_model.encode(dataset[0]['prompt'])
emb_B = st_model.encode(dataset[0]['B'])
cos = util.cos_sim(emb_prompt, emb_B).item()
cos


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7268.79it/s]


0.7658098936080933

Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?

In [13]:
import sys, os
sys.path.append(os.path.abspath("../.."))
from src.utils import OPTIONS, ap_at_3

In [14]:
def tfidf_top3(row):
    texts = [row['prompt']] + [row[o] for o in OPTIONS]
    vec = TfidfVectorizer()
    mat = vec.fit_transform(texts)
    sims = cosine_similarity(mat[0:1], mat[1:]).flatten()
    return [OPTIONS[i] for i in sims.argsort()[::-1][:3]]

prompt_embs = st_model.encode(dataset['prompt'], batch_size=64, show_progress_bar=True, convert_to_tensor=True)

option_embs = {o: st_model.encode(dataset[o], batch_size=64, show_progress_bar=True, convert_to_tensor=True) for o in OPTIONS}

def minilm_top3(idx):
    p = prompt_embs[idx]
    sims = [util.cos_sim(p, option_embs[o][idx]).item() for o in OPTIONS]
    return [OPTIONS[i] for i in np.argsort(sims)[::-1][:3]]

map3_minilm = 0.0
count_miss_hit = 0
n = len(dataset)

for i in range(n):
    row = dataset[i]
    correct = row['answer']

    top3_tf = tfidf_top3(row)
    top3_ml = minilm_top3(i)

    map3_minilm += ap_at_3(correct, top3_ml)

    if correct not in top3_tf and correct in top3_ml:
        count_miss_hit += 1

map3_minilm /= n
print(f"MAP@3 MiniLM: {map3_minilm:.4f}")
print(f"TF-IDF miss & MiniLM hit count: {count_miss_hit}")


Batches: 100%|██████████| 32/32 [00:02<00:00, 14.75it/s]


MAP@3 MiniLM: 0.4231
TF-IDF miss & MiniLM hit count: 564


### Zero-shot classification concepts 

Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option?

In [15]:
zs = pipeline("zero-shot-classification")

row1 = dataset[1]
labels = [row1['A'], row1['B'], row1['C']]
result_sm = zs(row1['prompt'], candidate_labels=labels)

result_sm['scores'][0]


[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 8415.73it/s]


0.45745086669921875

Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [16]:
result_sg = zs(row1['prompt'], candidate_labels=labels, multi_label=True)

sum_softmax = sum(result_sm['scores'])
sum_sigmoid = sum(result_sg['scores'])
abs(sum_softmax - sum_sigmoid)


0.9994903807710216

Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

In [ ]:
flan = pipeline("text2text-generation", model="google/flan-t5-small")

row0  = dataset[0]
query = (f"Question: {row0['prompt']}. Is the correct answer A: {row0['A']} or B: {row0['B']}? Answer with just the letter A or B.")

out = flan(query, max_new_tokens=5)
out

# It's giving error, turns out text2text-generation has been depreciated by hugging face pipeline and for text-generation the input is too big for it to work.